In [11]:
# Complete VGG16-based Image Classification with SHAP Integration

# Step 1: Install necessary libraries
!pip install tensorflow
!pip install shap
!pip install matplotlib

# Step 2: Import libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from google.colab import files
import io
import pandas as pd
import matplotlib.pyplot as plt
import os
from PIL import Image
import shap

# Step 3: Upload files
print("Please upload your dataset (images) in any format.")
uploaded = files.upload()

# Step 4: Load and preprocess data
def load_data(uploaded_files):
    image_list = []
    label_list = []

    for filename in uploaded_files.keys():
        # Check if the uploaded file is an image
        try:
            img = Image.open(io.BytesIO(uploaded_files[filename]))
            img = img.resize((32, 32))  # Resize to match VGG16 input size
            img_array = np.array(img) / 255.0  # Normalize the image
            image_list.append(img_array)
            label_list.append(filename)  # Use the filename as the label
        except Exception as e:
            print(f"Error loading {filename}: {e}")

    return np.array(image_list), np.array(label_list)

# Load the dataset
x_data, y_data = load_data(uploaded)

# Check the number of samples
print(f"Number of samples loaded: {len(x_data)}")

# Ensure there are enough samples
if len(x_data) == 0:
    raise ValueError("No images were loaded. Please upload valid image files.")

# Convert labels to categorical if there are enough samples
if len(x_data) > 1:
    class_names = np.unique(y_data)
    y_data = pd.get_dummies(y_data).values  # One-hot encoding
else:
    print("Not enough samples to create a validation set. Using all data for training.")
    y_data = np.array([[1]])  # Dummy label for a single sample

# Step 5: Build the model
def build_model():
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
    base_model.trainable = False  # Freeze the base model

    model = models.Sequential([
        base_model,
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(len(class_names), activation='softmax')  # Number of classes
    ])

    return model

model = build_model()
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Step 6: Train the model
if len(x_data) > 1:
    model.fit(x_data, y_data, epochs=10, batch_size=32, validation_split=0.2)
else:
    model.fit(x_data, y_data, epochs=10, batch_size=1)  # Train with a single sample

# Step 7: Evaluate the model
test_loss, test_acc = model.evaluate(x_data, y_data)
print(f'Test accuracy: {test_acc}')

# Step 8: SHAP integration for explainability
def explain_model(model, x_data):
    # Create a SHAP explainer
    explainer = shap.GradientExplainer(model, x_data)  # Use GradientExplainer for deep learning models
    shap_values = explainer.shap_values(x_data)

    # Visualize the SHAP values for the first image using color gradients
    for i in range(len(shap_values)):
        shap.image_plot(shap_values[i], x_data)

# Step 9: Visualize SHAP values
explain_model(model, x_data)

Please upload your dataset (images) in any format.


Saving brainscan.jpeg to brainscan.jpeg
Number of samples loaded: 1
Not enough samples to create a validation set. Using all data for training.
Epoch 1/10


ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("data:0", shape=(1, 32, 32), dtype=float32). Expected shape (None, 32, 32, 3), but input has incompatible shape (1, 32, 32)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(1, 32, 32), dtype=float32)
  • training=True
  • mask=None